# Liu2024 Multiscale Riemannian Fusion

**CLOSED DEVELOPMENT/CONFIRMATION ROUTE.** Do not rerun or tune the deleted grids from this notebook. The frozen result and STOP dispositions are in `AGENTS.md` section 2e. Execution fails closed.


# 1. Setup


In [ ]:
raise RuntimeError('CLOSED multiscale Riemann experiment: see AGENTS.md section 2e.')
import os, sys, json, glob, random, hashlib, builtins, platform, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy import signal, stats, linalg
from sklearn.covariance import OAS
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from pyriemann.tangentspace import TangentSpace
from pyriemann.utils.distance import distance_riemann, distance_logeuclid
from pyriemann.utils.mean import mean_covariance
from pyriemann.utils.geodesic import geodesic

print(f"Python: {sys.version.split()[0]} | Platform: {platform.platform()} | cwd: {Path.cwd()}")


# 2. Configuration
## 2.1 Domain Defaults

The 29-channel order drops recorded CPz (index 17). Views are the Cartesian product of four locked bands and seven locked temporal windows grouped into 1 s, 2 s, and 4 s scales.

## 2.2 CONFIG


In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # ------------------------------------------------------------------
    # Paths / run identity
    # ------------------------------------------------------------------
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-multiscale-riemann-fusion"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "covariance_cache_dir": str(WORKING_DIR / "artifacts" / "covariance_cache" / "multiscale_riemann_fusion"),
    "sjepa_cache_dir": str(WORKING_DIR / "artifacts" / "sjepa_embedding_cache" / "multiscale_riemann_fusion"),
    "experiment_name": "multiscale_riemann_fusion_full50_classical_5fold",
    "config_note": "Locked all-50 classical stratified five-fold within-subject evaluation.",

    # ------------------------------------------------------------------
    # Dataset and marker-relative trial processing
    # ------------------------------------------------------------------
    "subjects_to_use": None,
    "lv14_subject_ids": [1, 3, 7, 9, 10, 11, 14, 15, 17, 29, 31, 32, 37, 41],
    "channel_indices": list(range(17)) + list(range(18, 30)),
    "channel_names": ["Fp1","Fp2","Fz","F3","F4","F7","F8","FCz","FC3","FC4","FT7","FT8","Cz","C3","C4","T3","T4","CP3","CP4","TP7","TP8","Pz","P3","P4","T5","T6","Oz","O1","O2"],
    "spatial_mode": "full29",
    "filter_context": "cropped_mi",
    "marker_channel_index": 32,
    "onset_marker_value": 2,
    "onset_plausible_range": [800, 1300],
    "onset_fallback_sample": 1003,
    "sfreq": 500,
    "mi_window_s": [0.0, 4.0],

    # ------------------------------------------------------------------
    # Covariance views
    # ------------------------------------------------------------------
    "bands_hz": [[8, 12], [13, 20], [20, 30], [8, 30]],
    "temporal_scales": {"1s": {"length_s": 1.0, "starts_s": [0.0, 1.0, 2.0, 3.0]}, "2s": {"length_s": 2.0, "starts_s": [0.0, 1.0, 2.0]}, "4s": {"length_s": 4.0, "starts_s": [0.0]}},
    "filter_order": 4,
    "average_reference": True,
    "covariance_estimator": "oas",
    "fixed_shrinkage": 0.05,
    "covariance_downsample_factor": 4,
    "covariance_block_samples": 250,
    "trace_normalize": True,
    "tangent_metric": "riemann",
    "base_classifier": "tangent_lda",
    "tangent_pca_max_components": 8,
    "ridge_logistic_C": 0.1,
    "prototype_shrinkage": 0.25,
    "fixed_single_view": {"band_hz": [8, 30], "scale": "4s", "start_s": 0.0},

    # ------------------------------------------------------------------
    # Evaluation and inner-only selection
    # ------------------------------------------------------------------
    "outer_protocol": "stratified_5fold",
    "outer_folds": 5,
    "sensitivity_repeats": 10,
    "sensitivity_test_size": 0.40,
    "inner_folds": 4,
    "split_random_state": 2026,
    "selection_stability_lambda": 0.25,
    "top_k_views": 5,
    "collapse_threshold": 0.95,

    # ------------------------------------------------------------------
    # Branches and score fusion
    # ------------------------------------------------------------------
    "asymmetry_pairs": [["C3","C4"], ["FC3","FC4"], ["CP3","CP4"], ["P3","P4"]],
    "enable_sjepa": False,
    "enable_learned_stack": True,
    "methods_to_run": None,  # None preserves the full default method set.
    "stack_C": 0.05,
    "stack_l1_ratio": 0.5,

    # ------------------------------------------------------------------
    # Optional frozen S-JEPA provenance and extraction
    # ------------------------------------------------------------------
    "sjepa_model_id": "braindecode/signal-jepa_without-chans",
    "sjepa_model_revision": None,
    "sjepa_checkpoint_path": None,
    "sjepa_cache_policy": "refuse",
    "sjepa_sfreq": 128,
    "sjepa_bandpass_hz": [0.5, 40.0],
    "sjepa_window_s": [0.0, 4.0],
    "sjepa_hook": "feature_encoder",
    "sjepa_pooling": "mean",
    "sjepa_batch_size": 32,
    "device": "auto",

    # ------------------------------------------------------------------
    # Reproducibility and inference
    # ------------------------------------------------------------------
    "seed": 2026,
    "set_seed": True,
    "bootstrap_iterations": 10000,
    "bootstrap_seed": 202607,
}


In [ ]:
RECORDED_CH_NAMES = list(CONFIG["channel_names"])
RECORDED_CH_INDICES = list(CONFIG["channel_indices"])
MOTOR_MONTAGES = {
    "motor8": ["FC3", "FC4", "C3", "C4", "CP3", "CP4", "P3", "P4"],
    # 13 is honest: the three midline channels are supplemented by symmetric F3/F4.
    "motor13": ["F3", "F4", "FCz", "FC3", "FC4", "Cz", "C3", "C4", "CP3", "CP4", "Pz", "P3", "P4"],
}
def resolve_spatial_mode():
    mode = CONFIG["spatial_mode"]
    if mode in {"full29", "average_reference_subspace"}:
        names = RECORDED_CH_NAMES
    elif mode in MOTOR_MONTAGES:
        names = MOTOR_MONTAGES[mode]
    else:
        raise ValueError(f"Unknown spatial_mode: {mode}; use full29, motor8, motor13, or average_reference_subspace")
    positions = [RECORDED_CH_NAMES.index(name) for name in names]
    indices = [RECORDED_CH_INDICES[i] for i in positions]
    if mode == "average_reference_subspace":
        basis = linalg.helmert(len(names), full=False).T
        output_names = [f"helmert_{i + 1:02d}" for i in range(basis.shape[1])]
    else:
        basis, output_names = None, list(names)
    return indices, list(names), output_names, basis

SELECTED_CHANNEL_INDICES, SENSOR_CH_NAMES, CH_NAMES, SPATIAL_BASIS = resolve_spatial_mode()
VIEW_SPECS = [
    {"id": f"{scale}_{start:g}s_{lo:g}-{hi:g}Hz", "scale": scale, "start_s": float(start), "length_s": float(spec["length_s"]), "band_hz": [float(lo), float(hi)]}
    for scale, spec in CONFIG["temporal_scales"].items()
    for start in spec["starts_s"] for lo, hi in CONFIG["bands_hz"]
]
print(f"Subjects: {CONFIG['subjects_to_use'] or 'all 50'} | protocol: {CONFIG['outer_protocol']} | spatial: {CONFIG['spatial_mode']} ({len(CH_NAMES)}D) | filter: {CONFIG['filter_context']} | covariance: {CONFIG['covariance_estimator']} | classifier: {CONFIG['base_classifier']} | views: {len(VIEW_SPECS)} | S-JEPA: {CONFIG['enable_sjepa']}")


## 2.3 Artifact Creation and Logging Init


In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

def _safe_write_text(stream, text):
    try:
        stream.write(text)
    except UnicodeEncodeError:
        encoding = getattr(stream, "encoding", None) or "utf-8"
        stream.write(text.encode(encoding, errors="replace").decode(encoding, errors="replace"))

def _timestamped_print(*args, **kwargs):
    sep, end = kwargs.pop("sep", " "), kwargs.pop("end", "\n")
    flush, target = kwargs.pop("flush", False), kwargs.pop("file", None)
    message = sep.join(str(arg) for arg in args)
    stamped = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}" if message else ""
    for stream in ([target] if target is not None else [sys.stdout, _LOG_FILE_HANDLE]):
        _safe_write_text(stream, stamped + end)
    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print
config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2, allow_nan=False)
print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")


## 2.4 Reproducibility


In [ ]:
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.use_deterministic_algorithms(True, warn_only=True)
    except ImportError:
        pass

BASE_SEED = int(CONFIG["seed"])
if CONFIG["set_seed"]:
    seed_everything(BASE_SEED)
print(f"Seed initialized: {BASE_SEED}")


# 3. Load and Prepare Data
## 3.1 Data Loading Helpers

Each trial is sliced relative to its own marker before referencing or filtering. Filtering is performed on each trial separately; no operation sees adjacent trials.


In [ ]:
def selected_files():
    files = sorted(Path(CONFIG["source_extract_dir"]).glob("sub-*/sub-*_eeg.mat"))
    if not files:
        raise FileNotFoundError(CONFIG["source_extract_dir"])
    keep = CONFIG["subjects_to_use"]
    return [p for p in files if keep is None or int(p.parent.name.split("-")[1]) in set(keep)]

def load_subject(path):
    eeg = sio.loadmat(path)["eeg"][0, 0]
    raw = np.asarray(eeg["rawdata"], dtype=np.float64)
    y = np.asarray(eeg["label"]).ravel().astype(int)
    if set(np.unique(y)).issubset({1, 2}):
        y = y - 1
    marker = raw[:, CONFIG["marker_channel_index"], :]
    lo, hi = CONFIG["onset_plausible_range"]
    detected = []
    for trial_marker in marker:
        hits = np.flatnonzero(trial_marker == CONFIG["onset_marker_value"])
        valid = hits[(hits >= lo) & (hits <= hi)]
        detected.append(int(valid[0]) if len(valid) else -1)
    plausible = [x for x in detected if lo <= x <= hi]
    fallback = int(np.median(plausible)) if plausible else int(CONFIG["onset_fallback_sample"])
    onsets = np.asarray([x if lo <= x <= hi else fallback for x in detected], dtype=int)
    n = int(round((CONFIG["mi_window_s"][1] - CONFIG["mi_window_s"][0]) * CONFIG["sfreq"]))
    offset = int(round(CONFIG["mi_window_s"][0] * CONFIG["sfreq"]))
    if CONFIG["filter_context"] == "cropped_mi":
        trials = np.stack([raw[i, SELECTED_CHANNEL_INDICES, o + offset:o + offset + n] for i, o in enumerate(onsets)])
        if trials.shape[-1] != n:
            raise ValueError(f"Incomplete marker-relative MI trial in {path}")
    elif CONFIG["filter_context"] == "full_trial_then_crop":
        trials = raw[:, SELECTED_CHANNEL_INDICES, :].copy()
        if any(o + offset < 0 or o + offset + n > trials.shape[-1] for o in onsets):
            raise ValueError(f"MI crop falls outside full trial in {path}")
    else:
        raise ValueError("filter_context must be cropped_mi or full_trial_then_crop")
    sid = int(path.parent.name.split("-")[1])
    trial_ids = np.asarray([f"sub-{sid:02d}_trial-{i:03d}" for i in range(len(y))])
    return sid, trials, y, trial_ids, onsets

def make_outer_splits(y):
    if CONFIG["outer_protocol"] == "stratified_5fold":
        splitter = StratifiedKFold(n_splits=CONFIG["outer_folds"], shuffle=True, random_state=CONFIG["split_random_state"])
    elif CONFIG["outer_protocol"] == "repeated_60_40":
        splitter = StratifiedShuffleSplit(n_splits=CONFIG["sensitivity_repeats"], test_size=CONFIG["sensitivity_test_size"], random_state=CONFIG["split_random_state"])
    else:
        raise ValueError("outer_protocol must be stratified_5fold or repeated_60_40")
    return [(np.asarray(tr), np.asarray(te)) for tr, te in splitter.split(np.zeros(len(y)), y)]

def make_inner_splits(y_train, seed):
    splitter = StratifiedKFold(n_splits=CONFIG["inner_folds"], shuffle=True, random_state=seed)
    return [(np.asarray(tr), np.asarray(va)) for tr, va in splitter.split(np.zeros(len(y_train)), y_train)]

def assert_no_overlap(train_idx, test_idx):
    overlap = np.intersect1d(train_idx, test_idx)
    assert len(overlap) == 0, f"train/test overlap: {overlap.tolist()}"
    return True


## 3.2 Trial-Independent Covariance and Asymmetry Features

The covariance cache key covers every covariance-affecting setting. Cache payloads are accepted only if signature, labels, trial IDs, view IDs, and array dimensions match exactly.


In [ ]:
def covariance_signature():
    keys = ["sfreq", "mi_window_s", "marker_channel_index", "onset_marker_value", "onset_plausible_range", "onset_fallback_sample", "bands_hz", "temporal_scales", "asymmetry_pairs", "filter_order", "average_reference", "spatial_mode", "filter_context", "covariance_estimator", "fixed_shrinkage", "covariance_downsample_factor", "covariance_block_samples", "trace_normalize"]
    payload = {k: CONFIG[k] for k in keys}
    payload.update({"selected_channel_indices": SELECTED_CHANNEL_INDICES, "sensor_channel_names": SENSOR_CH_NAMES, "output_channel_names": CH_NAMES, "spatial_basis": None if SPATIAL_BASIS is None else np.round(SPATIAL_BASIS, 15).tolist()})
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:16]

def estimate_covariance(x):
    estimator = CONFIG["covariance_estimator"].lower()
    samples = x.T
    if estimator == "downsample_oas":
        samples = samples[::max(1, int(CONFIG["covariance_downsample_factor"]))]
        fitted = OAS(store_precision=False, assume_centered=True).fit(samples)
        cov, shrinkage = fitted.covariance_, float(fitted.shrinkage_)
    elif estimator == "block_covariance":
        block = int(CONFIG["covariance_block_samples"])
        blocks = [samples[start:start + block] for start in range(0, len(samples), block) if len(samples[start:start + block]) >= max(2, x.shape[0])]
        if not blocks:
            raise ValueError("No complete covariance block")
        fitted = [OAS(store_precision=False, assume_centered=True).fit(b) for b in blocks]
        cov, shrinkage = np.mean([f.covariance_ for f in fitted], axis=0), float(np.mean([f.shrinkage_ for f in fitted]))
    elif estimator == "fixed_shrinkage":
        shrinkage = float(CONFIG["fixed_shrinkage"])
        if not 0.0 <= shrinkage <= 1.0:
            raise ValueError("fixed_shrinkage must be in [0, 1]")
        empirical = samples.T @ samples / len(samples)
        cov = (1.0 - shrinkage) * empirical + shrinkage * np.trace(empirical) / empirical.shape[0] * np.eye(empirical.shape[0])
    elif estimator == "oas":
        fitted = OAS(store_precision=False, assume_centered=True).fit(samples)
        cov, shrinkage = fitted.covariance_, float(fitted.shrinkage_)
    else:
        raise ValueError("covariance_estimator must be oas, fixed_shrinkage, downsample_oas, or block_covariance")
    if CONFIG["trace_normalize"]:
        trace = np.trace(cov)
        if not np.isfinite(trace) or trace <= 0:
            raise ValueError("Invalid covariance trace")
        cov = cov / trace
    eig = np.linalg.eigvalsh(cov)
    eig = np.maximum(eig, np.finfo(float).eps)
    p = eig / eig.sum()
    effective_rank = float(np.exp(-np.sum(p * np.log(p))))
    condition = float(eig[-1] / eig[0])
    return cov, np.asarray([shrinkage, effective_rank, condition], dtype=float)

def extract_covariance_views(trials, onsets):
    n_trials, n_ch, _ = trials.shape
    covariance_dim = len(CH_NAMES)
    covs = np.empty((n_trials, len(VIEW_SPECS), covariance_dim, covariance_dim), dtype=np.float64)
    diagnostics = np.empty((n_trials, len(VIEW_SPECS), 3), dtype=np.float64)
    asym = np.empty((n_trials, len(VIEW_SPECS), len(CONFIG["asymmetry_pairs"])), dtype=np.float64)
    name_to_idx = {name: i for i, name in enumerate(SENSOR_CH_NAMES)}
    for trial_i, trial in enumerate(trials):
        must_reference = CONFIG["average_reference"] or CONFIG["spatial_mode"] == "average_reference_subspace"
        x = trial - trial.mean(axis=0, keepdims=True) if must_reference else trial.copy()
        filtered = {}
        for band in {tuple(v["band_hz"]) for v in VIEW_SPECS}:
            sos = signal.butter(CONFIG["filter_order"], band, btype="bandpass", fs=CONFIG["sfreq"], output="sos")
            z = signal.sosfiltfilt(sos, x, axis=-1)
            if CONFIG["filter_context"] == "full_trial_then_crop":
                offset = int(round(CONFIG["mi_window_s"][0] * CONFIG["sfreq"]))
                length = int(round((CONFIG["mi_window_s"][1] - CONFIG["mi_window_s"][0]) * CONFIG["sfreq"]))
                z = z[:, onsets[trial_i] + offset:onsets[trial_i] + offset + length]
            filtered[band] = z
        for view_i, view in enumerate(VIEW_SPECS):
            z = filtered[tuple(view["band_hz"])]
            start = int(round(view["start_s"] * CONFIG["sfreq"]))
            stop = start + int(round(view["length_s"] * CONFIG["sfreq"]))
            sensor_window = z[:, start:stop]
            window = SPATIAL_BASIS.T @ sensor_window if SPATIAL_BASIS is not None else sensor_window
            covs[trial_i, view_i], diagnostics[trial_i, view_i] = estimate_covariance(window)
            for pair_i, (left, right) in enumerate(CONFIG["asymmetry_pairs"]):
                if left not in name_to_idx or right not in name_to_idx:
                    asym[trial_i, view_i, pair_i] = np.nan
                else:
                    lp = np.mean(sensor_window[name_to_idx[left]] ** 2)
                    rp = np.mean(sensor_window[name_to_idx[right]] ** 2)
                    asym[trial_i, view_i, pair_i] = np.log(lp + 1e-12) - np.log(rp + 1e-12)
    return covs, asym, diagnostics

def source_fingerprint(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return {"name": Path(path).name, "size": Path(path).stat().st_size, "sha256": digest.hexdigest()}

def covariance_cache_path(sid):
    root = Path(CONFIG["covariance_cache_dir"]) / covariance_signature()
    root.mkdir(parents=True, exist_ok=True)
    return root / f"sub-{sid:02d}.npz"

def get_covariance_features(sid, trials, y, trial_ids, onsets, source_path):
    path = covariance_cache_path(sid)
    expected_cov_shape = (len(trials), len(VIEW_SPECS), len(CH_NAMES), len(CH_NAMES))
    expected_asym_shape = (len(trials), len(VIEW_SPECS), len(CONFIG["asymmetry_pairs"]))
    fingerprint = source_fingerprint(source_path)
    if path.exists():
        cached = np.load(path, allow_pickle=False)
        required = {"signature", "source_fingerprint_json", "labels", "trial_ids", "view_ids", "covariances", "asymmetry", "covariance_diagnostics"}
        valid = (required.issubset(cached.files) and str(cached["signature"].item()) == covariance_signature() and json.loads(str(cached["source_fingerprint_json"].item())) == fingerprint and np.array_equal(cached["labels"], y) and np.array_equal(cached["trial_ids"].astype(str), trial_ids.astype(str)) and np.array_equal(cached["view_ids"].astype(str), np.asarray([v["id"] for v in VIEW_SPECS])) and cached["covariances"].shape == expected_cov_shape and cached["asymmetry"].shape == expected_asym_shape and cached["covariance_diagnostics"].shape == (len(trials), len(VIEW_SPECS), 3) and np.issubdtype(cached["covariances"].dtype, np.floating) and np.issubdtype(cached["asymmetry"].dtype, np.floating) and np.all(np.isfinite(cached["covariances"])) and np.all(np.isfinite(cached["covariance_diagnostics"])))
        if not valid:
            raise ValueError(f"Incompatible covariance cache refused: {path}")
        return cached["covariances"], cached["asymmetry"], cached["covariance_diagnostics"]
    covs, asym, diagnostics = extract_covariance_views(trials, onsets)
    np.savez_compressed(path, signature=np.asarray(covariance_signature()), source_fingerprint_json=np.asarray(json.dumps(fingerprint, sort_keys=True)), labels=y, trial_ids=trial_ids, view_ids=np.asarray([v["id"] for v in VIEW_SPECS]), covariances=covs, asymmetry=asym, covariance_diagnostics=diagnostics)
    return covs, asym, diagnostics


## 3.3 Optional Frozen S-JEPA Features

The branch is lazy and disabled by default. A cache must exactly match model ID/revision, preprocessing, channel order, marker/window, hook/pooling, labels, and trial IDs. A mismatch is refused unless `sjepa_cache_policy` is explicitly `recompute`.


In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def sjepa_manifest(y, trial_ids):
    checkpoint = CONFIG["sjepa_checkpoint_path"]
    return {
        "model_id": CONFIG["sjepa_model_id"], "model_revision": CONFIG["sjepa_model_revision"],
        "checkpoint_path": checkpoint, "checkpoint_sha256": file_sha256(checkpoint) if checkpoint else None, "sfreq": CONFIG["sjepa_sfreq"],
        "bandpass_hz": CONFIG["sjepa_bandpass_hz"], "average_reference": CONFIG["average_reference"],
        "channel_order": CH_NAMES, "marker_channel_index": CONFIG["marker_channel_index"],
        "marker_value": CONFIG["onset_marker_value"], "mi_window_s": CONFIG["mi_window_s"],
        "sjepa_window_s": CONFIG["sjepa_window_s"], "hook": CONFIG["sjepa_hook"],
        "pooling": CONFIG["sjepa_pooling"], "labels": y.tolist(), "trial_ids": trial_ids.tolist(),
    }

def extract_sjepa_inline(trials):
    import torch
    from braindecode.models import SignalJEPA_PreLocal
    device = ("cuda" if torch.cuda.is_available() else "cpu") if CONFIG["device"] == "auto" else CONFIG["device"]
    processed = []
    for trial in trials:
        x = trial - trial.mean(axis=0, keepdims=True) if CONFIG["average_reference"] else trial.copy()
        lo, hi = CONFIG["sjepa_bandpass_hz"]
        sos = signal.butter(4, [lo, hi], btype="bandpass", fs=CONFIG["sfreq"], output="sos")
        x = signal.sosfiltfilt(sos, x, axis=-1)
        x = signal.resample_poly(x, CONFIG["sjepa_sfreq"], CONFIG["sfreq"], axis=-1)
        a, b = CONFIG["sjepa_window_s"]
        x = x[:, int(round(a * CONFIG["sjepa_sfreq"])):int(round(b * CONFIG["sjepa_sfreq"]))]
        processed.append(x.astype(np.float32))
    X = np.stack(processed)
    kwargs = dict(n_chans=len(CH_NAMES), chs_info=None, n_times=X.shape[-1], n_outputs=2)
    if CONFIG["sjepa_checkpoint_path"]:
        model = SignalJEPA_PreLocal(**kwargs)
        loaded = torch.load(CONFIG["sjepa_checkpoint_path"], map_location="cpu", weights_only=False)
        if isinstance(loaded, dict):
            for wrapper_key in ("state_dict", "model_state_dict", "model"):
                if wrapper_key in loaded and isinstance(loaded[wrapper_key], dict):
                    loaded = loaded[wrapper_key]
                    break
        if not isinstance(loaded, dict) or not loaded or not all(isinstance(k, str) for k in loaded):
            raise ValueError("S-JEPA checkpoint does not contain a valid state_dict")
        if all(k.startswith("module.") for k in loaded):
            loaded = {k[len("module."):]: v for k, v in loaded.items()}
        if all(k.startswith("_orig_mod.") for k in loaded):
            loaded = {k[len("_orig_mod."):]: v for k, v in loaded.items()}
        missing, unexpected = model.load_state_dict(loaded, strict=False)
        critical_missing = [k for k in missing if k.startswith("feature_encoder.")]
        if critical_missing or unexpected:
            raise ValueError(f"Incompatible S-JEPA checkpoint: critical_missing={critical_missing[:10]}, unexpected={unexpected[:10]}")
    else:
        if not CONFIG["sjepa_model_revision"]:
            raise ValueError("S-JEPA recomputation requires a non-null sjepa_model_revision or a hashed local checkpoint")
        model = SignalJEPA_PreLocal.from_pretrained(CONFIG["sjepa_model_id"], revision=CONFIG["sjepa_model_revision"], strict=False, **kwargs)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model = model.to(device).eval()
    captured = {}
    hook = getattr(model, CONFIG["sjepa_hook"]).register_forward_hook(lambda module, inputs, output: captured.__setitem__("value", output.detach()))
    features = []
    try:
        with torch.no_grad():
            for start in range(0, len(X), CONFIG["sjepa_batch_size"]):
                model(torch.from_numpy(X[start:start + CONFIG["sjepa_batch_size"]]).to(device))
                z = captured["value"]
                if CONFIG["sjepa_pooling"] == "mean" and z.ndim > 2:
                    z = z.mean(dim=1)
                elif CONFIG["sjepa_pooling"] == "flatten":
                    z = z.flatten(1)
                else:
                    if CONFIG["sjepa_pooling"] != "mean":
                        raise ValueError("sjepa_pooling must be mean or flatten")
                features.append(z.flatten(1).cpu().numpy())
    finally:
        hook.remove()
    return np.concatenate(features).astype(np.float32)

def get_sjepa_features(sid, trials, y, trial_ids):
    if not CONFIG["enable_sjepa"]:
        return None
    path = Path(CONFIG["sjepa_cache_dir"]) / f"sub-{sid:02d}.npz"
    expected = sjepa_manifest(y, trial_ids)
    if path.exists():
        cached = np.load(path, allow_pickle=False)
        required = {"manifest_json", "features"}
        observed = json.loads(str(cached["manifest_json"].item())) if required.issubset(cached.files) else None
        features = cached["features"] if "features" in cached.files else None
        feature_valid = features is not None and features.ndim == 2 and features.shape[0] == len(trials) and features.shape[1] > 0 and np.issubdtype(features.dtype, np.floating) and np.all(np.isfinite(features))
        if observed == expected and feature_valid:
            return features.astype(np.float32)
        if CONFIG["sjepa_cache_policy"] != "recompute":
            raise ValueError(f"S-JEPA cache provenance mismatch refused: {path}")
    elif CONFIG["sjepa_cache_policy"] == "refuse":
        raise FileNotFoundError(f"Required S-JEPA cache is missing and policy=refuse: {path}")
    elif CONFIG["sjepa_cache_policy"] != "recompute":
        raise ValueError("sjepa_cache_policy must be refuse or recompute")
    if not CONFIG["sjepa_checkpoint_path"] and not CONFIG["sjepa_model_revision"]:
        raise ValueError("S-JEPA recomputation requires a non-null revision or local checkpoint")
    features = extract_sjepa_inline(trials)
    if features.ndim != 2 or features.shape[0] != len(trials) or features.shape[1] == 0 or not np.issubdtype(features.dtype, np.floating) or not np.all(np.isfinite(features)):
        raise ValueError(f"Invalid recomputed S-JEPA feature matrix: {features.shape}, {features.dtype}")
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, features=features, manifest_json=np.asarray(json.dumps(expected, sort_keys=True)))
    return features


# 4. Model
## 4.1 Scalar Decision-Score Branches

Every view and branch emits one class-oriented scalar score. Tangent coefficients are never concatenated across views. Tangent references, scalers, classifiers, and asymmetry/S-JEPA transforms are refit in every applicable inner fold.


In [ ]:
def oriented_decision_score(model, X):
    score = np.asarray(model.decision_function(X), dtype=float)
    if score.ndim == 2:
        score = score[:, list(model.classes_).index(1)] - score[:, list(model.classes_).index(0)]
    elif list(model.classes_)[-1] != 1:
        score = -score
    return score.ravel()

def robust_margin_scale(train_score):
    train_score = np.asarray(train_score, dtype=float)
    scale = float(np.median(np.abs(train_score)))
    if not np.isfinite(scale) or scale < 1e-8:
        scale = float(np.sqrt(np.mean(train_score ** 2)))
    if not np.isfinite(scale) or scale < 1e-8:
        raise ValueError("Degenerate fitted training margins")
    return scale

def method_complexity(n_channels, n_train):
    classifier = CONFIG["base_classifier"]
    tangent_dim = n_channels * (n_channels + 1) // 2
    if classifier == "tangent_pca_logistic":
        fitted_dim = min(int(CONFIG["tangent_pca_max_components"]), tangent_dim, n_train - 2)
        parameters = fitted_dim + 1
    elif classifier == "tangent_lda":
        fitted_dim, parameters = tangent_dim, tangent_dim + 1
    else:
        fitted_dim, parameters = n_channels, 2 * tangent_dim
    return {"classifier": classifier, "covariance_dimension": n_channels, "tangent_dimension": tangent_dim, "fitted_feature_dimension": fitted_dim, "approximate_parameters": parameters, "prototype_shrinkage": float(CONFIG["prototype_shrinkage"]) if classifier == "mdm_riemann_prototype" else 0.0}

def fit_view_score(covs, y, train_idx, query_idx):
    classifier = CONFIG["base_classifier"]
    train_covs, query_covs, y_train = covs[train_idx], covs[query_idx], y[train_idx]
    if classifier in {"mdm_riemann", "mdm_logeuclid", "mdm_riemann_prototype"}:
        metric = "logeuclid" if classifier == "mdm_logeuclid" else "riemann"
        prototypes = [mean_covariance(train_covs[y_train == label], metric=metric) for label in [0, 1]]
        if classifier == "mdm_riemann_prototype":
            alpha = float(CONFIG["prototype_shrinkage"])
            if not 0.0 <= alpha <= 1.0:
                raise ValueError("prototype_shrinkage must be in [0, 1]")
            global_mean = mean_covariance(train_covs, metric="riemann")
            prototypes = [geodesic(proto, global_mean, alpha, metric="riemann") for proto in prototypes]
        distance = distance_logeuclid if metric == "logeuclid" else distance_riemann
        score = lambda rows: np.asarray([distance(row, prototypes[0]) - distance(row, prototypes[1]) for row in rows])
        train_score = score(train_covs)
        return score(query_covs) / robust_margin_scale(train_score)
    tangent = TangentSpace(metric=CONFIG["tangent_metric"]).fit(train_covs)
    X_train, X_query = tangent.transform(train_covs), tangent.transform(query_covs)
    scaler = StandardScaler().fit(X_train)
    X_train, X_query = scaler.transform(X_train), scaler.transform(X_query)
    if classifier == "tangent_lda":
        model = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto").fit(X_train, y_train)
    elif classifier == "tangent_pca_logistic":
        n_components = min(int(CONFIG["tangent_pca_max_components"]), X_train.shape[1], len(train_idx) - 2)
        if n_components < 1:
            raise ValueError("No valid fold-local PCA components")
        pca = PCA(n_components=n_components, svd_solver="full").fit(X_train)
        X_train, X_query = pca.transform(X_train), pca.transform(X_query)
        model = LogisticRegression(C=float(CONFIG["ridge_logistic_C"]), solver="liblinear", random_state=BASE_SEED).fit(X_train, y_train)
    else:
        raise ValueError("base_classifier must be tangent_lda, tangent_pca_logistic, mdm_riemann, mdm_logeuclid, or mdm_riemann_prototype")
    train_score = oriented_decision_score(model, X_train)
    return oriented_decision_score(model, X_query) / robust_margin_scale(train_score)

def fit_feature_score(features, y, train_idx, query_idx):
    valid = np.all(np.isfinite(features[train_idx]), axis=0)
    if not valid.any():
        raise ValueError("No finite branch features")
    if not np.all(np.isfinite(features[query_idx][:, valid])):
        raise ValueError("Nonfinite query values in training-selected feature columns")
    scaler = StandardScaler().fit(features[train_idx][:, valid])
    model = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto").fit(scaler.transform(features[train_idx][:, valid]), y[train_idx])
    train_score = oriented_decision_score(model, scaler.transform(features[train_idx][:, valid]))
    return oriented_decision_score(model, scaler.transform(features[query_idx][:, valid])) / robust_margin_scale(train_score)

def stability_scores(view_oof, y_train, inner_splits):
    result = []
    for view_i in range(view_oof.shape[1]):
        fold_ba = [balanced_accuracy_score(y_train[va], view_oof[va, view_i] >= 0) for _, va in inner_splits]
        result.append(float(np.mean(fold_ba) - CONFIG["selection_stability_lambda"] * np.std(fold_ba)))
    return np.asarray(result)

def choose_stable_views(view_oof, y_train, inner_splits, top_k):
    stability = stability_scores(view_oof, y_train, inner_splits)
    selected = np.argsort(-stability)[:min(top_k, len(stability))]
    raw_weights = np.maximum(stability[selected] - 0.5, 0.0) + 1e-6
    return selected, raw_weights / raw_weights.sum(), stability

def equal_score_fusion(*scores):
    arrays = [np.asarray(s, dtype=float) for s in scores if s is not None]
    if not arrays:
        raise ValueError("No scores supplied to fusion")
    return np.mean(np.column_stack(arrays), axis=1)

def collapse_diagnostic(score):
    pred = np.asarray(score) >= 0
    fraction = float(max(np.mean(pred), np.mean(~pred)))
    return {"majority_prediction_fraction": fraction, "collapsed": bool(fraction >= CONFIG["collapse_threshold"]), "score_sd": float(np.std(score))}


## 4.2 Shared Inner OOF Construction and Hierarchical Stack

All branches use the same persisted inner split list. The elastic-net stack sees only inner OOF scalar base scores. Its conservative hyperparameters are fixed unless the optional tiny grid is enabled; that grid is evaluated only inside the outer training set.


In [ ]:
def asymmetry_scale_features(asymmetry, scale):
    indices = [i for i, view in enumerate(VIEW_SPECS) if view["scale"] == scale]
    return np.nanmean(asymmetry[:, indices, :], axis=1)

def build_inner_oof(covs, asymmetry, sjepa, y_train, inner_splits, view_indices=None, need_asymmetry=True):
    n, n_views = len(y_train), covs.shape[1]
    view_indices = list(range(n_views)) if view_indices is None else sorted(set(view_indices))
    view_oof = np.full((n, n_views), np.nan)
    scales = list(CONFIG["temporal_scales"])
    asym_oof = np.full((n, len(scales)), np.nan) if need_asymmetry else None
    sjepa_oof = np.full(n, np.nan) if sjepa is not None else None
    for inner_train, inner_valid in inner_splits:
        for view_i in view_indices:
            view_oof[inner_valid, view_i] = fit_view_score(covs[:, view_i], y_train, inner_train, inner_valid)
        if need_asymmetry:
            for scale_i, scale in enumerate(scales):
                asym_features = asymmetry_scale_features(asymmetry, scale)
                asym_oof[inner_valid, scale_i] = fit_feature_score(asym_features, y_train, inner_train, inner_valid)
        if sjepa is not None:
            sjepa_oof[inner_valid] = fit_feature_score(sjepa, y_train, inner_train, inner_valid)
    if not np.all(np.isfinite(view_oof[:, view_indices])) or (asym_oof is not None and not np.all(np.isfinite(asym_oof))) or (sjepa_oof is not None and not np.all(np.isfinite(sjepa_oof))):
        raise ValueError("Incomplete inner OOF scores")
    return view_oof, asym_oof, sjepa_oof

def fit_elastic_stack(X_oof, y_train, X_test):
    C, ratio = CONFIG["stack_C"], CONFIG["stack_l1_ratio"]
    scaler = StandardScaler().fit(X_oof)
    model = LogisticRegression(solver="saga", penalty="elasticnet", C=C, l1_ratio=ratio, max_iter=5000, random_state=BASE_SEED).fit(scaler.transform(X_oof), y_train)
    return oriented_decision_score(model, scaler.transform(X_test)), {"penalty": "elasticnet", "C": C, "l1_ratio": ratio, "tuning": "fixed_locked"}


# 5. Training
## 5.1 Leakage-Safe Outer-Fold Runner

The outer test indices are not passed to view selection, score normalization, hyperparameter choice, or stack fitting. They are transformed once after all outer-training decisions are locked.


In [ ]:
def run_outer_fold(sid, fold_id, covs, asymmetry, sjepa, y, train_idx, test_idx):
    seed_everything(BASE_SEED + sid * 1000 + fold_id)
    no_overlap = assert_no_overlap(train_idx, test_idx)
    y_train = y[train_idx]
    inner_splits = make_inner_splits(y_train, BASE_SEED + sid * 1000 + fold_id)
    for inner_train, inner_valid in inner_splits:
        assert_no_overlap(inner_train, inner_valid)
    requested = set(CONFIG["methods_to_run"] or [])
    restricted = bool(requested)
    scales = list(CONFIG["temporal_scales"])
    fixed_id = next(i for i, v in enumerate(VIEW_SPECS) if v["band_hz"] == [float(x) for x in CONFIG["fixed_single_view"]["band_hz"]] and v["scale"] == CONFIG["fixed_single_view"]["scale"] and v["start_s"] == CONFIG["fixed_single_view"]["start_s"])
    need_stable = not restricted or bool(requested & {"riemann_stable_topk", "riemann_asymmetry_fusion", "riemann_sjepa_fusion", "full_parameter_free_fusion", "hierarchical_elastic_net_stack"})
    need_nested = not restricted or "nested_best_single_view" in requested
    need_asymmetry = not restricted or bool(requested & {"asymmetry", "riemann_asymmetry_fusion", "full_parameter_free_fusion", "hierarchical_elastic_net_stack"}) or any(m.startswith("asymmetry_scale_") for m in requested)
    needed_scales = set(scales) if not restricted or "riemann_equal_multiscale" in requested else set()
    needed_scales.update(m.removeprefix("riemann_scale_") for m in requested if m.startswith("riemann_scale_"))
    if "riemann_equal_short_scales" in requested:
        needed_scales.update({"1s", "2s"})
    view_indices = {fixed_id} if not restricted or "fixed_single_view" in requested else set()
    view_indices.update(i for i, v in enumerate(VIEW_SPECS) if v["scale"] in needed_scales)
    if need_stable or need_nested:
        view_indices.update(range(covs.shape[1]))
    train_covs, train_asym = covs[train_idx], asymmetry[train_idx]
    train_sjepa = sjepa[train_idx] if sjepa is not None else None
    view_oof, asym_oof, sjepa_oof = build_inner_oof(train_covs, train_asym, train_sjepa, y_train, inner_splits, view_indices, need_asymmetry)
    view_test = np.full((len(test_idx), covs.shape[1]), np.nan)
    for view_i in sorted(view_indices):
        view_test[:, view_i] = fit_view_score(covs[:, view_i], y, train_idx, test_idx)
    asym_test = np.column_stack([fit_feature_score(asymmetry_scale_features(asymmetry, scale), y, train_idx, test_idx) for scale in scales]) if need_asymmetry else None
    sjepa_test = fit_feature_score(sjepa, y, train_idx, test_idx) if sjepa is not None else None

    selected, weights, stability = choose_stable_views(view_oof, y_train, inner_splits, CONFIG["top_k_views"]) if (need_stable or need_nested) else (np.array([], dtype=int), np.array([]), np.full(covs.shape[1], np.nan))
    best = int(np.nanargmax(stability)) if need_nested else None
    oof_branches, test_branches = {}, {}
    if not restricted or "fixed_single_view" in requested:
        oof_branches["fixed_single_view"], test_branches["fixed_single_view"] = view_oof[:, fixed_id], view_test[:, fixed_id]
    if need_nested:
        oof_branches["nested_best_single_view"], test_branches["nested_best_single_view"] = view_oof[:, best], view_test[:, best]
    for scale in needed_scales:
        idx = [i for i, v in enumerate(VIEW_SPECS) if v["scale"] == scale]
        oof_branches[f"riemann_scale_{scale}"] = view_oof[:, idx].mean(axis=1)
        test_branches[f"riemann_scale_{scale}"] = view_test[:, idx].mean(axis=1)
    if not restricted or "riemann_equal_multiscale" in requested:
        oof_branches["riemann_equal_multiscale"] = equal_score_fusion(*[oof_branches[f"riemann_scale_{scale}"] for scale in scales])
        test_branches["riemann_equal_multiscale"] = equal_score_fusion(*[test_branches[f"riemann_scale_{scale}"] for scale in scales])
    if not restricted or "riemann_equal_short_scales" in requested:
        oof_branches["riemann_equal_short_scales"] = equal_score_fusion(oof_branches["riemann_scale_1s"], oof_branches["riemann_scale_2s"])
        test_branches["riemann_equal_short_scales"] = equal_score_fusion(test_branches["riemann_scale_1s"], test_branches["riemann_scale_2s"])
    if need_stable:
        oof_branches["riemann_stable_topk"] = view_oof[:, selected] @ weights
        test_branches["riemann_stable_topk"] = view_test[:, selected] @ weights
    for scale_i, scale in (enumerate(scales) if need_asymmetry else []):
        oof_branches[f"asymmetry_scale_{scale}"], test_branches[f"asymmetry_scale_{scale}"] = asym_oof[:, scale_i], asym_test[:, scale_i]
    if need_asymmetry:
        oof_branches["asymmetry"] = equal_score_fusion(*[oof_branches[f"asymmetry_scale_{scale}"] for scale in scales])
        test_branches["asymmetry"] = equal_score_fusion(*[test_branches[f"asymmetry_scale_{scale}"] for scale in scales])
    if not restricted or "riemann_asymmetry_fusion" in requested:
        oof_branches["riemann_asymmetry_fusion"] = equal_score_fusion(oof_branches["riemann_stable_topk"], oof_branches["asymmetry"])
        test_branches["riemann_asymmetry_fusion"] = equal_score_fusion(test_branches["riemann_stable_topk"], test_branches["asymmetry"])
    if sjepa_oof is not None:
        oof_branches["sjepa"], test_branches["sjepa"] = sjepa_oof, sjepa_test
        oof_branches["riemann_sjepa_fusion"] = equal_score_fusion(oof_branches["riemann_stable_topk"], sjepa_oof)
        test_branches["riemann_sjepa_fusion"] = equal_score_fusion(test_branches["riemann_stable_topk"], sjepa_test)
        oof_branches["full_parameter_free_fusion"] = equal_score_fusion(oof_branches["riemann_stable_topk"], oof_branches["asymmetry"], sjepa_oof)
        test_branches["full_parameter_free_fusion"] = equal_score_fusion(test_branches["riemann_stable_topk"], test_branches["asymmetry"], sjepa_test)
    if CONFIG["enable_learned_stack"] and (not restricted or "hierarchical_elastic_net_stack" in requested):
        stack_names = ["riemann_stable_topk", "asymmetry"] + (["sjepa"] if sjepa_oof is not None else [])
        stack_score, stack_params = fit_elastic_stack(np.column_stack([oof_branches[n] for n in stack_names]), y_train, np.column_stack([test_branches[n] for n in stack_names]))
        test_branches["hierarchical_elastic_net_stack"] = stack_score
    else:
        stack_names, stack_params = [], None

    records = []
    emitted_branches = {m: s for m, s in test_branches.items() if not restricted or m in requested}
    for method, score in emitted_branches.items():
        pred = (score >= 0).astype(int)
        records.append({"subject_id": sid, "fold_id": fold_id, "method": method, "train_indices": train_idx.tolist(), "test_indices": test_idx.tolist(), "y_true": y[test_idx].tolist(), "y_pred": pred.tolist(), "decision_score": np.asarray(score).tolist(), "accuracy": float(accuracy_score(y[test_idx], pred)), "balanced_accuracy": float(balanced_accuracy_score(y[test_idx], pred)), "confusion_matrix": confusion_matrix(y[test_idx], pred, labels=[0, 1]).tolist(), "selected_best_view": VIEW_SPECS[best]["id"] if best is not None else None, "selected_topk_views": [VIEW_SPECS[i]["id"] for i in selected], "selected_topk_weights": weights.tolist(), "view_stability_scores": stability.tolist(), "stack_base_branches": stack_names, "stack_parameters": stack_params, "method_complexity": method_complexity(covs.shape[-1], len(train_idx)), "branch_correlations_inner_oof": pd.DataFrame(oof_branches).corr().to_dict(), "collapse_diagnostics": collapse_diagnostic(score), "no_train_test_overlap": no_overlap, "outer_test_used_for_selection": False, "shared_inner_splits": [{"train": a.tolist(), "valid": b.tolist()} for a, b in inner_splits]})
    return records


## 5.2 Run All Subjects


In [ ]:
print("=" * 72)
print(json.dumps(CONFIG, indent=2, sort_keys=True))
print("=" * 72)

FOLD_RESULTS, FOLD_FAILURES, SPLIT_RECORDS, SUBJECTS, COVARIANCE_DIAGNOSTICS = [], [], {}, [], []
inventory = []
split_indices_path = ARTIFACT_DIR / "split_indices.json"
for path in selected_files():
    sid, trials, y, trial_ids, onsets = load_subject(path)
    SUBJECTS.append(sid)
    inventory.append({"subject_id": sid, "n_trials": len(y), "class_0": int(np.sum(y == 0)), "class_1": int(np.sum(y == 1)), "source_path": str(path)})
    outer_splits = make_outer_splits(y)
    SPLIT_RECORDS[str(sid)] = [{"fold_id": i, "train_indices": tr.tolist(), "test_indices": te.tolist(), "no_overlap": assert_no_overlap(tr, te), "inner_split_seed": BASE_SEED + sid * 1000 + i, "inner_folds": CONFIG["inner_folds"]} for i, (tr, te) in enumerate(outer_splits)]
    with open(split_indices_path, "w") as f:
        json.dump({"protocol": CONFIG["outer_protocol"], "outer_split_seed": CONFIG["split_random_state"], "inner_seed_formula": "BASE_SEED + subject_id * 1000 + fold_id", "subjects": SPLIT_RECORDS}, f, indent=2, allow_nan=False)
    covs, asymmetry, covariance_diagnostics = get_covariance_features(sid, trials, y, trial_ids, onsets, path)
    COVARIANCE_DIAGNOSTICS.append({"subject_id": sid, "spatial_mode": CONFIG["spatial_mode"], "filter_context": CONFIG["filter_context"], "covariance_estimator": CONFIG["covariance_estimator"], "n_channels": int(covs.shape[-1]), "shrinkage_mean": float(np.mean(covariance_diagnostics[..., 0])), "shrinkage_sd": float(np.std(covariance_diagnostics[..., 0])), "effective_rank_mean": float(np.mean(covariance_diagnostics[..., 1])), "effective_rank_sd": float(np.std(covariance_diagnostics[..., 1])), "condition_median": float(np.median(covariance_diagnostics[..., 2])), "condition_p95": float(np.percentile(covariance_diagnostics[..., 2], 95)), "cache_signature": covariance_signature()})
    sjepa = get_sjepa_features(sid, trials, y, trial_ids)
    for fold_id, (train_idx, test_idx) in enumerate(outer_splits):
        try:
            FOLD_RESULTS.extend(run_outer_fold(sid, fold_id, covs, asymmetry, sjepa, y, train_idx, test_idx))
        except Exception as exc:
            failure = {"subject_id": sid, "fold_id": fold_id, "error_type": type(exc).__name__, "error": str(exc)}
            FOLD_FAILURES.append(failure)
            print(f"FOLD FAILURE: {failure}")
            if CONFIG["outer_protocol"] == "stratified_5fold":
                warnings.warn("A primary-protocol fold failed; no substitute prediction was produced.")
    print(f"sub-{sid:02d}: completed {len(outer_splits)} outer splits")

subject_inventory_path = ARTIFACT_DIR / "subject_inventory.csv"
pd.DataFrame(inventory).to_csv(subject_inventory_path, index=False)
with open(split_indices_path, "w") as f:
    json.dump({"protocol": CONFIG["outer_protocol"], "outer_split_seed": CONFIG["split_random_state"], "inner_seed_formula": "BASE_SEED + subject_id * 1000 + fold_id", "subjects": SPLIT_RECORDS}, f, indent=2, allow_nan=False)


# 6. Results
## 6.1 Subject-Level Pooled OOF Metrics

Five-fold subject metrics require complete exactly-once coverage of every trial; any fold failure invalidates that subject for every method. Repeated 60/40 sensitivity metrics average split-level balanced accuracy within subject, so repeated test appearances are not pooled or unequally weighted.


In [ ]:
SUBJECT_METRICS, INVALID_SUBJECT_METHODS = [], []
INVALID_SUBJECTS = sorted({f["subject_id"] for f in FOLD_FAILURES})
inventory_by_subject = {row["subject_id"]: row for row in inventory}
for sid in SUBJECTS:
    methods = sorted({r["method"] for r in FOLD_RESULTS if r["subject_id"] == sid})
    for method in methods:
        rows = [r for r in FOLD_RESULTS if r["subject_id"] == sid and r["method"] == method]
        if CONFIG["outer_protocol"] == "stratified_5fold":
            indices = [idx for row in rows for idx in row["test_indices"]]
            expected = list(range(40))
            subject_failed = any(f["subject_id"] == sid for f in FOLD_FAILURES)
            complete = inventory_by_subject[sid]["n_trials"] == 40 and not subject_failed and len(rows) == CONFIG["outer_folds"] and sorted(indices) == expected and len(set(indices)) == len(expected)
            if not complete:
                INVALID_SUBJECT_METHODS.append({"subject_id": sid, "method": method, "reason": "incomplete_exactly_once_outer_oof", "n_folds": len(rows), "n_predictions": len(indices), "subject_had_fold_failure": subject_failed})
                continue
            y_true = np.concatenate([r["y_true"] for r in rows])
            y_pred = np.concatenate([r["y_pred"] for r in rows])
            value = float(balanced_accuracy_score(y_true, y_pred))
            metric_name = "complete_exactly_once_pooled_outer_oof_balanced_accuracy"
        else:
            if len(rows) != CONFIG["sensitivity_repeats"]:
                INVALID_SUBJECT_METHODS.append({"subject_id": sid, "method": method, "reason": "incomplete_repeated_60_40_splits", "n_folds": len(rows), "expected_folds": CONFIG["sensitivity_repeats"]})
                continue
            value = float(np.mean([r["balanced_accuracy"] for r in rows]))
            metric_name = "mean_repeat_level_60_40_balanced_accuracy"
        SUBJECT_METRICS.append({"subject_id": sid, "method": method, "subject_balanced_accuracy": value, "subject_metric_name": metric_name, "n_outer_splits": len(rows), "valid_complete_subject_method": True})

def bootstrap_mean_ci(values, iterations, seed):
    values = np.asarray(values, float)
    rng = np.random.default_rng(seed)
    means = np.mean(rng.choice(values, size=(iterations, len(values)), replace=True), axis=1)
    return [float(x) for x in np.quantile(means, [0.025, 0.975])]

def paired_comparison(method_a, method_b):
    table = pd.DataFrame(SUBJECT_METRICS).pivot(index="subject_id", columns="method", values="subject_balanced_accuracy")
    if method_a not in table or method_b not in table:
        return None
    paired = table[[method_a, method_b]].dropna()
    delta = (paired[method_a] - paired[method_b]).to_numpy()
    try:
        wilcoxon_result = stats.wilcoxon(delta, alternative="two-sided")
        statistic, pvalue = float(wilcoxon_result.statistic), float(wilcoxon_result.pvalue)
    except ValueError:
        statistic, pvalue = 0.0, 1.0
    return {"method_a": method_a, "method_b": method_b, "n_subjects": len(delta), "mean_paired_delta": float(np.mean(delta)), "bootstrap_95_ci": bootstrap_mean_ci(delta, CONFIG["bootstrap_iterations"], CONFIG["bootstrap_seed"]), "wilcoxon_statistic": statistic, "wilcoxon_pvalue": pvalue, "inference_unit": "subject"}

method_summary = {}
for method in sorted({r["method"] for r in SUBJECT_METRICS}):
    values = [r["subject_balanced_accuracy"] for r in SUBJECT_METRICS if r["method"] == method]
    method_summary[method] = {"mean_subject_balanced_accuracy": float(np.mean(values)), "subject_metric_name": next(r["subject_metric_name"] for r in SUBJECT_METRICS if r["method"] == method), "bootstrap_95_ci_over_subjects": bootstrap_mean_ci(values, CONFIG["bootstrap_iterations"], CONFIG["bootstrap_seed"]), "n_valid_complete_subjects": len(values)}

fusion_method = "full_parameter_free_fusion" if CONFIG["enable_sjepa"] else "riemann_asymmetry_fusion"
PLANNED_COMPARISONS = {
    "fusion_vs_riemann": paired_comparison(fusion_method, "riemann_stable_topk"),
    "fusion_vs_sjepa": paired_comparison(fusion_method, "sjepa") if CONFIG["enable_sjepa"] else None,
}
metric_label = "complete exactly-once subject pooled outer-OOF balanced accuracy, averaged over valid subjects" if CONFIG["outer_protocol"] == "stratified_5fold" else "mean repeat-level 60/40 balanced accuracy per subject, averaged over valid subjects"
GLOBAL_METRICS = {"primary_metric": metric_label, "outer_protocol": CONFIG["outer_protocol"], "methods": method_summary, "planned_paired_comparisons": PLANNED_COMPARISONS, "n_fold_failures": len(FOLD_FAILURES), "invalid_subjects_with_any_fold_failure": INVALID_SUBJECTS, "n_invalid_subject_methods": len(INVALID_SUBJECT_METHODS), "invalid_subject_methods": INVALID_SUBJECT_METHODS, "outer_test_used_for_selection": False}


## 6.2 Performance Visualizations


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

performance_plot_path = ARTIFACT_DIR / "multiscale_riemann_fusion_subject_performance.png"
plot_data = pd.DataFrame(SUBJECT_METRICS)
headline = [m for m in ["fixed_single_view", "riemann_equal_short_scales", "riemann_stable_topk", "asymmetry", "sjepa", fusion_method, "hierarchical_elastic_net_stack"] if m in set(plot_data["method"])]
pivot = plot_data[plot_data["method"].isin(headline)].pivot(index="subject_id", columns="method", values="subject_balanced_accuracy")
ax = pivot.plot(kind="bar", figsize=(16, 6), width=0.85)
ax.axhline(0.5, color="black", linestyle="--", linewidth=1)
ax.set_ylabel(GLOBAL_METRICS["primary_metric"])
ax.set_title(f"Liu2024 multiscale fusion: {CONFIG['outer_protocol']}")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3)
plt.tight_layout(); plt.savefig(performance_plot_path, dpi=160); plt.close()


## 6.3 Experiment Summary


In [ ]:
print("Primary metric: subject-level pooled outer-OOF BA, then mean across subjects")
for method, metrics in GLOBAL_METRICS["methods"].items():
    print(f"{method}: {100 * metrics['mean_subject_balanced_accuracy']:.2f}%")
print("Planned subject-level comparisons:", json.dumps(PLANNED_COMPARISONS, indent=2))
print(f"Fold failures: {len(FOLD_FAILURES)} (never replaced by constant or fallback models)")


## 6.4 Save Artifacts


In [ ]:
cv_results_path = ARTIFACT_DIR / "cv_results.json"
subject_metrics_path = ARTIFACT_DIR / "subject_metrics.json"
global_metrics_path = ARTIFACT_DIR / "global_metrics.json"
fold_failures_path = ARTIFACT_DIR / "fold_failures.json"
predictions_path = ARTIFACT_DIR / "predictions.csv"
selection_path = ARTIFACT_DIR / "selection_diagnostics.json"
covariance_diagnostics_path = ARTIFACT_DIR / "covariance_diagnostics.json"

def json_sanitize(value):
    if isinstance(value, dict):
        return {str(k): json_sanitize(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_sanitize(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_sanitize(value.tolist())
    if isinstance(value, np.generic):
        return json_sanitize(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value

with open(cv_results_path, "w") as f: json.dump(json_sanitize(FOLD_RESULTS), f, indent=2, allow_nan=False)
with open(subject_metrics_path, "w") as f: json.dump(json_sanitize(SUBJECT_METRICS), f, indent=2, allow_nan=False)
with open(global_metrics_path, "w") as f: json.dump(json_sanitize(GLOBAL_METRICS), f, indent=2, allow_nan=False)
with open(fold_failures_path, "w") as f: json.dump(json_sanitize(FOLD_FAILURES), f, indent=2, allow_nan=False)
pd.DataFrame([{"subject_id": r["subject_id"], "fold_id": r["fold_id"], "method": r["method"], "trial_index": idx, "y_true": yt, "y_pred": yp, "decision_score": score} for r in FOLD_RESULTS for idx, yt, yp, score in zip(r["test_indices"], r["y_true"], r["y_pred"], r["decision_score"])]).to_csv(predictions_path, index=False)
with open(selection_path, "w") as f: json.dump(json_sanitize([{"subject_id": r["subject_id"], "fold_id": r["fold_id"], "method": r["method"], "selected_best_view": r["selected_best_view"], "selected_topk_views": r["selected_topk_views"], "selected_topk_weights": r["selected_topk_weights"], "view_stability_scores": r["view_stability_scores"], "method_complexity": r["method_complexity"], "branch_correlations_inner_oof": r["branch_correlations_inner_oof"], "collapse_diagnostics": r["collapse_diagnostics"], "outer_test_used_for_selection": r["outer_test_used_for_selection"]} for r in FOLD_RESULTS]), f, indent=2, allow_nan=False)
with open(covariance_diagnostics_path, "w") as f: json.dump(json_sanitize(COVARIANCE_DIAGNOSTICS), f, indent=2, allow_nan=False)

run_metadata = {"run_id": RUN_ID, "artifact_dir": str(ARTIFACT_DIR), "experiment_name": CONFIG["experiment_name"], "config_note": CONFIG["config_note"], "subjects": SUBJECTS, "spatial_mode": CONFIG["spatial_mode"], "filter_context": CONFIG["filter_context"], "covariance_estimator": CONFIG["covariance_estimator"], "base_classifier": CONFIG["base_classifier"], "n_channels": len(CH_NAMES), "channel_names": CH_NAMES, "sensor_channel_names": SENSOR_CH_NAMES, "spatial_basis": None if SPATIAL_BASIS is None else SPATIAL_BASIS.tolist(), "view_specs": VIEW_SPECS, "seed": BASE_SEED, "split_seed": CONFIG["split_random_state"], "global_metrics": GLOBAL_METRICS, "leakage_assertions": {"all_outer_splits_no_overlap": all(x["no_overlap"] for rows in SPLIT_RECORDS.values() for x in rows), "outer_test_used_for_selection": False, "shared_inner_split_lists": True}, "performance_artifacts": {"cv_results": str(cv_results_path), "subject_metrics": str(subject_metrics_path), "global_metrics": str(global_metrics_path), "fold_failures": str(fold_failures_path), "predictions": str(predictions_path), "selection_diagnostics": str(selection_path), "covariance_diagnostics": str(covariance_diagnostics_path), "split_indices": str(split_indices_path), "subject_inventory": str(subject_inventory_path), "subject_performance_plot": str(performance_plot_path)}}
run_metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(run_metadata_path, "w") as f: json.dump(json_sanitize(run_metadata), f, indent=2, allow_nan=False)
print(f"CV results saved to:      {cv_results_path}")
print(f"Subject metrics saved to: {subject_metrics_path}")
print(f"Global metrics saved to:  {global_metrics_path}")
print(f"Run metadata saved to:    {run_metadata_path}")
print(f"\nAll artifacts in: {ARTIFACT_DIR}")
try:
    _LOG_FILE_HANDLE.close()
except Exception:
    pass
